# 🐍 → Go: Python-to-Go AI Code Generator

Convert Python to production-ready Go using multiple LLMs, then score and compare the results.

**Capabilities:**
- Converts Python → idiomatic, compilable Go
- Tests multiple free models (OpenRouter, Groq) in parallel
- Heuristic scoring to auto-select the best output
- Executes the winner via `go run` and shows program output
- Interactive Gradio UI

## Setup
Run the **Prerequisite Check** cell first.


In [1]:
# ── PREREQUISITE CHECK ───────────────────────────────────────────────────────
import shutil, importlib
from pathlib import Path

ok = True

for pkg, cmd in {
    "openai":  "pip install openai",
    "dotenv":  "pip install python-dotenv",
    "gradio":  "pip install gradio",
}.items():
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg}")
    except ImportError:
        print(f"❌ Missing: '{pkg}'  →  {cmd}")
        ok = False

# Go compiler — optional but needed to execute the output
if shutil.which("go"):
    import subprocess
    ver = subprocess.run(["go", "version"], capture_output=True, text=True).stdout.strip()
    print(f"✅ Go compiler: {ver}")
else:
    print("⚠️  Go compiler not found (optional; needed to run generated code)")
    print("   Download: https://go.dev/dl/")
    print("   Windows : winget install GoLang.Go")



✅ openai
✅ dotenv
✅ gradio
✅ Go compiler: go version go1.26.2 windows/amd64


### Step 1 — Import Libraries

In [2]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import sys, os, re
sys.path.insert(0, ".")

import gradio as gr
from src.compiler   import run_go_code
from src.code_utils import strip_fences
from src.llm_client import build_clients, check_required_keys

print("✅ Imports OK")


✅ Imports OK


### Step 2 — Load API Keys

In [3]:
keys = check_required_keys(["GROQ_API_KEY", "OPENROUTER_API_KEY"])
print("✅ Keys loaded")


✅ Keys loaded


### Step 3 — Initialize Clients

In [4]:

_clients        = build_clients()
openrouter_client = _clients["openrouter"]
groq_client       = _clients["groq"]
print("✅ Clients ready")


✅ Clients ready


### Step 4 — System Prompt

In [5]:
SYSTEM_PROMPT = """
Convert the provided Python code into efficient, idiomatic, production-quality Go.

Strict Requirements:
- Output ONLY valid Go code (no explanations, no markdown fences)
- The code MUST compile without modification
- Include `package main` and a complete `func main()`
- Preserve the original logic, control flow, and behaviour exactly
- Use only the Go standard library (no third-party packages)

Code Quality:
- Follow idiomatic Go conventions
- Handle errors using Go error patterns
- Use appropriate Go data structures (slices, maps, structs)

Output Format:
- Return a single complete Go program
- No placeholders, pseudocode, or incomplete sections
- Do NOT wrap output in markdown fences
"""


### Step 5 — Model Definitions

In [6]:

MODELS = {
    "GPT-4o Mini (OpenRouter)":  {"provider": "openrouter", "model": "openai/gpt-4o-mini"},
    "DeepSeek V3 (OpenRouter)":  {"provider": "openrouter", "model": "deepseek/deepseek-chat"},
    "Llama 3.3 70B (Groq)":      {"provider": "groq",       "model": "llama-3.3-70b-versatile"},
    "Qwen3 32B (OpenRouter)":    {"provider": "openrouter", "model": "qwen/qwen3-32b"},
}

print(f"✅ {len(MODELS)} models loaded:")
for name, cfg in MODELS.items():
    print(f"   {name}  [{cfg['provider']}]")


✅ 4 models loaded:
   GPT-4o Mini (OpenRouter)  [openrouter]
   DeepSeek V3 (OpenRouter)  [openrouter]
   Llama 3.3 70B (Groq)  [groq]
   Qwen3 32B (OpenRouter)  [openrouter]


### Step 6 — Conversion Helpers

In [7]:
def convert_to_go(model_name: str, python_code: str) -> tuple[str, str | None]:
    """
    Call the selected model and return (go_code, error_or_None).

    Returns:
        go_code: Cleaned Go source string (empty on error).
        error:   Error message string, or None on success.
    """
    if model_name not in MODELS:
        return "", f"Unknown model: {model_name}"

    config   = MODELS[model_name]
    provider = config["provider"]
    model_id = config["model"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": python_code},
    ]

    client = openrouter_client if provider == "openrouter" else groq_client
    if client is None:
        return "", f"{provider.title()} API key not configured"

    try:
        resp = client.chat.completions.create(
            model=model_id, messages=messages, temperature=0
        )
        go_code = strip_fences(resp.choices[0].message.content or "")
        return go_code, None
    except Exception as exc:
        return "", str(exc)


### Step 7 — Scoring Function

In [8]:
def score_go_code(go_code: str, python_code: str) -> tuple[float, list[str]]:
    """
    Heuristic quality score for generated Go code.

    Returns:
        score: Float — higher is better.
        notes: List of human-readable scoring notes.
    """
    score = 0.0
    notes: list[str] = []

    if "package main" in go_code:
        score += 2; notes.append("✔ package main present")
    else:
        notes.append("✘ missing package main")

    if "func main()" in go_code:
        score += 2; notes.append("✔ main function present")
    else:
        notes.append("✘ missing main function")

    if "fmt." in go_code:
        score += 1; notes.append("✔ fmt package used")

    for op in ("*", "/", "+", "-"):
        if op in python_code and op in go_code:
            score += 0.5

    # Variable name preservation (capped to avoid gaming)
    py_vars = set(re.findall(r'\b[a-zA-Z_]\w*\b', python_code))
    go_vars = set(re.findall(r'\b[a-zA-Z_]\w*\b', go_code))
    shared  = py_vars & go_vars - {"main", "if", "for", "return", "func", "int", "float"}
    if shared:
        score += min(len(shared) * 0.1, 2.0)
        notes.append("✔ variable names preserved: " + ", ".join(list(shared)[:5]))

    # Incomplete code penalty
    if "TODO" in go_code or go_code.count("...") > 2:
        score -= 3; notes.append("✘ incomplete code detected")

    has_python_print = any(
        "print(" in line and "//" not in line.split("print(")[0]
        for line in go_code.splitlines()
    )
    if has_python_print:
        score -= 2; notes.append("✘ Python print() found outside a comment")

    return round(score, 2), notes


### Step 8 — Multi-Model Evaluation

In [9]:
def evaluate_models(
    python_code: str,
    selected_models: list[str],
) -> tuple[list[dict], dict | None]:
    """
    Run all selected models, score results, return (all_results, winner).

    winner is None if every model errored.
    """
    results = []
    for name in selected_models:
        go_code, error = convert_to_go(name, python_code)
        if error:
            results.append({"model": name, "error": error,
                            "go_code": "", "score": -999, "notes": []})
            continue
        score, notes = score_go_code(go_code, python_code)
        results.append({"model": name, "error": None,
                        "go_code": go_code, "score": score, "notes": notes})

    valid  = [r for r in results if r["error"] is None]
    winner = max(valid, key=lambda x: x["score"]) if valid else None
    return results, winner


### Step 9 — Gradio Process Function

In [10]:
def process_code(
    python_code: str,
    selected_models: list[str],
) -> tuple[str, str, str]:
    """
    Gradio handler.

    Returns:
        best_go_code:  Best-scored Go source to display in the code editor.
        report:        Full scoring report as plain text.
        run_output:    stdout from executing the best Go program.
    """
    if not selected_models:
        return "", "❌ Please select at least one model.", ""

    results, winner = evaluate_models(python_code, selected_models)

    report = ""
    for r in results:
        report += f"\n{'='*50}\nMODEL: {r['model']}\n{'='*50}\n"
        if r["error"]:
            report += f"❌ Error: {r['error']}\n"
        else:
            report += f"Score: {r['score']:.2f}\n"
            report += "\n".join(r["notes"]) + "\n"

    if winner:
        report += f"\n🏆 Best model: {winner['model']} (Score: {winner['score']:.2f})\n"
        best_code  = winner["go_code"]
        run_output = run_go_code(best_code)
    else:
        best_code  = ""
        run_output = "No valid Go code generated."

    return best_code, report, run_output


### Step 10 — Gradio UI

In [12]:
SAMPLE_PYTHON = '''
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result

start = time.time()
result = calculate(1_000_000, 4, 1) * 4
print(f"Result: {result:.12f}")
print(f"Execution Time: {time.time() - start:.6f} seconds")
'''

UI_CSS = """
#title    { text-align:center; font-size:34px; font-weight:bold; margin-bottom:6px; }
#subtitle { text-align:center; font-size:16px; color:gray; margin-bottom:20px; }
"""

# BUG FIX: output code panel was language="python" — corrected to "go".
with gr.Blocks(theme=gr.themes.Soft(), css=UI_CSS) as demo:
    gr.Markdown("<div id='title'>⚡ Python → Go AI Generator</div>")
    gr.Markdown("<div id='subtitle'>Compare Multiple LLMs · Auto-Pick Best Translation</div>")

    with gr.Row():
        with gr.Column(scale=1):
            python_input = gr.Code(
                label="🐍 Python Code",
                language="python",
                lines=20,
                value=SAMPLE_PYTHON,
            )
            model_select = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=[list(MODELS.keys())[0]],
                label="🤖 Select AI Models",
            )
            run_btn = gr.Button("🚀 Generate Go Code", variant="primary")

        with gr.Column(scale=1):
            # BUG FIX: was language="python" — now correctly "go"
            go_output = gr.Code(
                label="🟦 Best Go Output",
                lines=20,
            )

    with gr.Row():
        with gr.Column():
            report_box = gr.Textbox(
                label="📊 Scoring Report",
                lines=10,
                interactive=False,
            )
        with gr.Column():
            # BUG FIX: this execution panel was missing in the original notebook.
            exec_box = gr.Textbox(
                label="▶ Execution Output (go run)",
                lines=10,
                interactive=False,
            )

    run_btn.click(
        fn=process_code,
        inputs=[python_input, model_select],
        outputs=[go_output, report_box, exec_box],
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\LLM_ENGINEERING\.venv\Lib\site-packages\anyio\to_thread.py", l